In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"hrrajath","key":"6a6ed28ecf8a98bbea8479f38c1830bc"}'}

In [2]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign -p "/content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN"
!unzip "/content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/gtsrb-german-traffic-sign.zip" -d "/content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN"

Streaming output truncated to the last 5000 lines.
  inflating: /content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/train/5/00005_00053_00010.png  
  inflating: /content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/train/5/00005_00053_00011.png  
  inflating: /content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/train/5/00005_00053_00012.png  
  inflating: /content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/train/5/00005_00053_00013.png  
  inflating: /content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/train/5/00005_00053_00014.png  
  inflating: /content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/train/5/00005_00053_00015.png  
  inflating: /content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/train/5/00005_00053_00016.png  
  inflating: /content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/train/5/00005_00053_00017.png  
  inflating: /content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/train/5/00005_00053_00018.png  
  inflating: /content/drive/MyDrive/Traffic_Sign_Project/G

In [11]:
import pandas as pd
import os
import shutil

In [12]:
csv_path = "/content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/Test.csv"
df = pd.read_csv(csv_path)

flat_test_dir = "/content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/Test"
organized_dir = "/content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/Test_Organized"

In [ ]:
for class_id in df['ClassId'].unique():
    os.makedirs(os.path.join(organized_dir, str(class_id)), exist_ok=True)

for _, row in df.iterrows():
    fname = os.path.basename(row['Path'])
    label = str(row['ClassId'])

    src = os.path.join(flat_test_dir, fname)
    dst = os.path.join(organized_dir, label, fname)

    if os.path.exists(src):
        shutil.copy(src, dst)


In [15]:
train_dir = r"/content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/Train"
test_dir = r"/content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/Test_Organized"

In [16]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = 64
batch_size = 32

# Apply data augmentation to training set :
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical'
)


Found 39209 images belonging to 43 classes.
Found 12630 images belonging to 43 classes.


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

num_classes = 43

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_size, img_size, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dense(num_classes, activation='softmax')
])


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator
)


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.3569 - loss: 2.2429

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1226/1226 ━━━━━━━━━━━━━━━━━━━━ 224s 179ms/step - accuracy: 0.3571 - loss: 2.2422 - val_accuracy: 0.8153 - val_loss: 0.5386
Epoch 2/10
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 258s 179ms/step - accuracy: 0.8260 - loss: 0.5386 - val_accuracy: 0.9215 - val_loss: 0.2506
Epoch 3/10
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 258s 176ms/step - accuracy: 0.9043 - loss: 0.2971 - val_accuracy: 0.9373 - val_loss: 0.2107
Epoch 4/10
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 229s 187ms/step - accuracy: 0.9350 - loss: 0.2099 - val_accuracy: 0.9434 - val_loss: 0.2170
Epoch 5/10
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 244s 199ms/step - accuracy: 0.9505 - loss: 0.1599 - val_accuracy: 0.9564 - val_loss: 0.1696
Epoch 6/10
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 229s 187ms/step - accuracy: 0.9564 - loss: 0.1359 - val_accuracy: 0.9549 - val_loss: 0.1655
Epoch 7/10
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 219s 179ms/step - accuracy: 0.9638 - loss: 0.1137 - val_accuracy: 0.9550 - val_loss: 0.2110
Epoch 8/10
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 216s 176ms/step - accuracy: 0.9

In [19]:
model.save('/content/drive/MyDrive/Traffic_Sign_Project/Models/cnn_model.h5')

Model_evaluation-

In [20]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

# Prepare test data
test_dir = "/content/drive/MyDrive/Traffic_Sign_Project/GTSRB_CNN/Test_Organized"
img_size = 64
batch_size = 32

test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


Found 12630 images belonging to 43 classes.


In [21]:
predictions = model.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


395/395 ━━━━━━━━━━━━━━━━━━━━ 41s 102ms/step


In [22]:
from sklearn.metrics import classification_report,confusion_matrix
class_labels = list(test_generator.class_indices.keys())
print(classification_report(y_true, y_pred, target_names=class_labels))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99        60
           1       0.98      0.99      0.98       720
          10       1.00      1.00      1.00       660
          11       0.87      0.99      0.93       420
          12       1.00      0.97      0.98       690
          13       0.98      1.00      0.99       720
          14       0.99      1.00      1.00       270
          15       0.95      1.00      0.98       210
          16       0.91      1.00      0.95       150
          17       1.00      0.92      0.96       360
          18       0.98      0.86      0.92       390
          19       0.80      1.00      0.89        60
           2       0.98      0.99      0.98       750
          20       0.66      0.99      0.79        90
          21       0.86      0.73      0.79        90
          22       1.00      0.69      0.82       120
          23       0.98      0.92      0.95       150
          24       0.98    

In [26]:
cm = confusion_matrix(y_true, y_pred)
print(cm)

[[ 60   0   0 ...   0   0   0]
 [  0 713   0 ...   2   0   0]
 [  0   0 657 ...   0   0   2]
 ...
 [  0   0   0 ... 443   2   0]
 [  0   0   0 ...   5 436   1]
 [  0   0   0 ...   0   0 480]]
